# 04 - Feature and target preparation

Builds the model-ready arrays from the extracted per-pixel monthly time series.

**Inputs** - the per-pixel monthly series exported in notebook 02, the fire-reference
inventory labels, and the project land-cover product.

**Outputs** - `X_scaled.npy` (608,922 x 108 x 52), `y_burned.npy`, `targets_aux.npz`
(severity, persistence, recovery gap, recurrence), `scaler.pkl`, `splits.json`.

**Role** - the last preparation stage before training in notebook 05.

In [ ]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules


if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

In [ ]:
import importlib
import config
importlib.reload(config)
config.ensure_dirs()
config.set_seed()
config.print_summary()

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 1. Load raw data and selected features

In [ ]:
# Use the same compact schema as notebook 03. The default float64 parse and
# the copy made by sorting cannot fit comfortably in memory for this file.
_float_cols = [c for c in (config.ALL_FEATURES + config.LC_FEATURES)
               if c not in ("pixel_id", "year", "month", "burned")]
dtypes = {c: "float32" for c in _float_cols}
dtypes.update({"pixel_id": "int32", "pixel_type": "category",
               "year": "int16", "month": "int8", "burned": "int8"})

_header = pd.read_csv(config.FILES["raw_timeseries"], nrows=0).columns
dtypes = {c: dtype for c, dtype in dtypes.items() if c in _header}

df = pd.read_csv(config.FILES["raw_timeseries"], dtype=dtypes)
df = df.sort_values(["pixel_id", "year", "month"], ignore_index=True)

# Rename category labels without expanding pixel_type to an object array.
_old_categories = list(df["pixel_type"].cat.categories)
_new_categories = [config.PIXEL_TYPE_LEGACY_MAP.get(c, c)
                   for c in _old_categories]
if _new_categories != _old_categories:
    if len(set(_new_categories)) == len(_new_categories):
        df["pixel_type"] = df["pixel_type"].cat.rename_categories(
            dict(zip(_old_categories, _new_categories)))
    else:
        _unique_categories = sorted(set(_new_categories))
        _category_remap = np.array(
            [_unique_categories.index(c) for c in _new_categories],
            dtype=np.int16)
        _old_codes = df["pixel_type"].cat.codes.to_numpy()
        _new_codes = np.full(len(_old_codes), -1, dtype=np.int16)
        _valid_codes = _old_codes >= 0
        _new_codes[_valid_codes] = _category_remap[_old_codes[_valid_codes]]
        df["pixel_type"] = pd.Categorical.from_codes(
            _new_codes, categories=_unique_categories)

with open(config.FILES["selected_features"]) as f:
    selected_features = json.load(f)["selected"]

print(f"Memory:   {df.memory_usage(deep=True).sum() / 1e9:.1f} GB")
print(f"Rows:     {len(df):,}")
print(f"Pixels:   {df['pixel_id'].nunique():,}")
print(f"Features: {len(selected_features)}")
print(f"  Spectral: {len(selected_features) - len(config.LC_FEATURES)}")
print(f"  LC:       {len(config.LC_FEATURES)}")
print(f"Pixel types: "
      f"{df.groupby('pixel_id', observed=True)['pixel_type'].first().value_counts().to_dict()}")


## 2. Reshape to [N, T, C]

Each pixel becomes a fixed-length sequence of `EXPECTED_T` months.
Padding is applied where a pixel has fewer rows. A boolean mask records
which positions are real data and which are padding - the L-TAE uses
this mask in its attention weighting.

In [ ]:
month_list  = config.get_month_list()
month_index = {(year, month): i for i, (year, month) in enumerate(month_list)}
T = config.EXPECTED_T

# df is sorted by pixel_id, so unique IDs are in tensor order.
pixel_ids = np.unique(df["pixel_id"].to_numpy())
N = len(pixel_ids)
C = len(selected_features)

# A failed or interrupted re-run may have left these large arrays in memory.
for _stale_name in ("X", "y", "mask"):
    globals().pop(_stale_name, None)

X    = np.full((N, T, C), np.nan, dtype=np.float32)
y    = np.zeros((N, T), dtype=np.float32)
mask = np.zeros((N, T), dtype=bool)

# Vectorized row-to-(pixel, month) mapping. Keeping features column-wise avoids
# materializing a second rows-by-features block of roughly the same size as X.
print("Mapping rows to tensor indices...")
_years = df["year"].to_numpy(dtype=np.int32)
_months = df["month"].to_numpy(dtype=np.int32)
_linear_months = _years * 12 + (_months - 1)
_month_keys = np.array(
    [year * 12 + (month - 1) for year, month in month_list], dtype=np.int32)
_month_low = int(_month_keys.min())
_month_high = int(_month_keys.max())
_month_lut = np.full(_month_high - _month_low + 1, -1, dtype=np.int16)
_month_lut[_month_keys - _month_low] = np.arange(T, dtype=np.int16)

t_of_row = np.full(len(df), -1, dtype=np.int16)
_in_range = ((_linear_months >= _month_low) &
             (_linear_months <= _month_high))
t_of_row[_in_range] = _month_lut[_linear_months[_in_range] - _month_low]
i_of_row = np.searchsorted(
    pixel_ids, df["pixel_id"].to_numpy()).astype(np.int32)

row_ok = t_of_row >= 0
i_ok = i_of_row[row_ok]
t_ok = t_of_row[row_ok]
del (_years, _months, _linear_months, _month_keys, _month_lut,
     _in_range, i_of_row, t_of_row)

print(f"Filling tensors ({row_ok.sum():,} of {len(df):,} rows in window)...")
_raw_features = [name for name in selected_features if name in df.columns]
_derived_features = set(config.DELTA_FEATURES + config.HARMONIC_FEATURES)
_unknown_features = [name for name in selected_features
                     if name not in df.columns and name not in _derived_features]
if _unknown_features:
    raise KeyError(
        f"Selected features are neither raw nor derivable: {_unknown_features}")

for _feature_name in _raw_features:
    _feature_i = selected_features.index(_feature_name)
    _values = df[_feature_name].to_numpy(dtype=np.float32, copy=False)
    X[i_ok, t_ok, _feature_i] = _values[row_ok]

y[i_ok, t_ok] = df["burned"].to_numpy(dtype=np.float32, copy=False)[row_ok]
mask[i_ok, t_ok] = True
if _raw_features:
    del _values

# Match notebook 07b/inference: delta at t is previous month minus current
# month. t=0 remains NaN and is handled by the imputation cell below.
for _band_name in config.DELTA_SOURCE_INDICES:
    _delta_name = f"delta_{_band_name}"
    if _delta_name not in selected_features:
        continue
    if _band_name not in selected_features:
        raise KeyError(f"Cannot derive {_delta_name}: {_band_name} is not selected")
    _source_i = selected_features.index(_band_name)
    _target_i = selected_features.index(_delta_name)
    X[:, 1:, _target_i] = (
        X[:, :-1, _source_i] - X[:, 1:, _source_i])

_calendar_months = np.array(
    [month for _, month in month_list], dtype=np.float32)
if "month_cos" in selected_features:
    X[:, :, selected_features.index("month_cos")] = np.cos(
        2 * np.pi * _calendar_months / 12.0)[None, :]
if "month_sin" in selected_features:
    X[:, :, selected_features.index("month_sin")] = np.sin(
        2 * np.pi * _calendar_months / 12.0)[None, :]

print("Building metadata...")
meta_df_tmp = (df.groupby("pixel_id", observed=True)["pixel_type"]
                 .first().reindex(pixel_ids).reset_index())
meta_df_tmp["pixel_id"] = meta_df_tmp["pixel_id"].astype(int)
meta_df_tmp["pixel_type"] = meta_df_tmp["pixel_type"].astype(str)
meta = meta_df_tmp.to_dict("records")

print(f"\nX shape:       {X.shape}")
print(f"y shape:       {y.shape}")
print(f"mask coverage: {100 * mask.mean():.1f}%")
print(f"Derived features: "
      f"{[name for name in selected_features if name in _derived_features]}")


## 3. NaN imputation by calendar month

Missing spectral values are replaced with the mean for their calendar
month, so seasonality is preserved. Land-cover features are 0/1 by
construction and should already contain no NaN; the loop below is
safe on them because their means are meaningful frequencies.

In [ ]:
calendar_month = np.array([m for _, m in month_list])

for cm in range(1, 13):
    t_idx = np.where(calendar_month == cm)[0]
    if len(t_idx) == 0:
        continue
    block = X[:, t_idx, :]
    flat = block.reshape(-1, C)
    col_means = np.nanmean(flat, axis=0)
    inds = np.where(np.isnan(flat))
    flat[inds] = np.take(col_means, inds[1])
    X[:, t_idx, :] = flat.reshape(block.shape)

# Check and repair remaining NaNs in pixel chunks. A whole-tensor
# np.isnan(X) temporary would itself occupy several gigabytes.
_impute_chunk = 25_000
_has_nan = any(
    np.isnan(X[_start:_start + _impute_chunk]).any()
    for _start in range(0, N, _impute_chunk)
)
if _has_nan:
    _global_sum = np.zeros(C, dtype=np.float64)
    _global_count = np.zeros(C, dtype=np.int64)
    for _start in range(0, N, _impute_chunk):
        _flat = X[_start:_start + _impute_chunk].reshape(-1, C)
        _global_sum += np.nansum(_flat, axis=0, dtype=np.float64)
        _global_count += np.sum(~np.isnan(_flat), axis=0)
    if np.any(_global_count == 0):
        _bad = np.asarray(selected_features)[_global_count == 0].tolist()
        raise ValueError(f"Features contain no finite values: {_bad}")
    _global_means = (_global_sum / _global_count).astype(np.float32)

    for _start in range(0, N, _impute_chunk):
        _flat = X[_start:_start + _impute_chunk].reshape(-1, C)
        _nan_rows, _nan_cols = np.where(np.isnan(_flat))
        _flat[_nan_rows, _nan_cols] = _global_means[_nan_cols]

assert not any(
    np.isnan(X[_start:_start + _impute_chunk]).any()
    for _start in range(0, N, _impute_chunk)
), "NaN imputation incomplete"
print("Imputation complete; no NaN remain.")


## 4. Analytical target construction

Four per-pixel indicators are derived. All are defined so that higher values
indicate greater restoration need.

* **severity** - mean negative standardized NBR anomaly over the pixel's
  documented fire months.
* **persistence** - fraction of post-fire months whose standardized NBR anomaly
  stays below -0.5.
* **recovery gap** - end-of-record spectral departure from the reference state,
  measured between seasonally aligned 12-month windows.
* **recurrence** - number of documented burned months, scaled by 5 and capped at 1.

recovery gap is defined only for pixels with at least one documented fire month
and is set to 0 elsewhere. Pixels that never burned have nothing to recover from,
so background NBR drift over the record must not be read as a recovery deficit.

These are spectral indicators derived from the same Sentinel-2 series used as
model input. They are not independently validated measures of ecological condition.

In [ ]:
# df still contains the unscaled source NBR. Reuse the row mapping from tensor
# construction instead of loading and sorting the 36 GB CSV a second time.
nbr_raw = np.full((N, T), np.nan, dtype=np.float32)
nbr_raw[i_ok, t_ok] = df["NBR"].to_numpy(
    dtype=np.float32, copy=False)[row_ok]

for cm in range(1, 13):
    t_idx = np.where(calendar_month == cm)[0]
    if len(t_idx) == 0:
        continue
    block = nbr_raw[:, t_idx]
    col_mean = np.nanmean(block)
    block[np.isnan(block)] = col_mean
    nbr_raw[:, t_idx] = block

print(f"NBR raw shape: {nbr_raw.shape}")
print(f"NBR raw range: [{nbr_raw.min():.3f}, {nbr_raw.max():.3f}]")


In [ ]:
def compute_targets(X, y, nbr_raw, feature_names,
                    baseline_months=12, recent_months=12):
    """Compute four-dimensional analytical targets.

    All scores are calibrated so higher = greater restoration need.

    severity:     mean negative NBR z-score during fire months (0-1)
    persistence:  fraction of post-fire months with NBR z-score below -0.5
    recovery_gap: relative distance from the pre-disturbance NBR baseline,
                  ONLY for pixels that experienced fire. no_fire pixels
                  receive recovery_gap=0.
    recurrence:   fire count normalised by 5 (capped at 1)

    recovery_gap is 0 for pixels with no documented fire: a pixel that never
    burned has nothing to recover from, so background spectral drift over the
    record must not be read as a recovery deficit.
    """
    N, T, C = X.shape
    try:
        nbr_z_idx = feature_names.index("NBR_zscore")
    except ValueError:
        nbr_z_idx = feature_names.index("NBR")

    severity     = np.zeros(N, dtype=np.float32)
    persistence  = np.zeros(N, dtype=np.float32)
    recovery_gap = np.zeros(N, dtype=np.float32)
    recurrence   = np.zeros(N, dtype=np.float32)

    for i in range(N):
        nbr_z         = X[i, :, nbr_z_idx]
        burned_months = np.where(y[i] == 1)[0]

        recurrence[i] = float(np.clip(len(burned_months) / 5.0, 0, 1))

        # === No documented fire: all four indicators stay at 0 ===
        # recovery_gap, severity, persistence all stay 0.
        if len(burned_months) == 0:
            continue
        # =========================================================

        # recovery_gap: only meaningful when there was a disturbance
        baseline_nbr = float(np.nanmean(nbr_raw[i, :baseline_months]))
        recent_nbr   = float(np.nanmean(nbr_raw[i, -recent_months:]))
        gap          = (baseline_nbr - recent_nbr) / (abs(baseline_nbr) + 1e-6)
        recovery_gap[i] = float(np.clip(gap, 0, 1))

        # severity: mean negative NBR z-score during fire months
        drops       = -nbr_z[burned_months]
        severity[i] = float(np.clip(drops.mean() / 5.0, 0, 1))

        # persistence: fraction of post-fire months with NBR z-score < -0.5
        last_fire = burned_months[-1]
        post_fire = nbr_z[last_fire:]
        if len(post_fire) > 1:
            persistence[i] = float((post_fire < -0.5).mean())

    return {
        "recurrence":   recurrence,
        "severity":     severity,
        "persistence":  persistence,
        "recovery_gap": recovery_gap,
    }


targets = compute_targets(X, y, nbr_raw, selected_features)

print("Target statistics (mean / std / max):")
for name, arr in targets.items():
    print(f"  {name:14s}  {arr.mean():.3f}  {arr.std():.3f}  {arr.max():.3f}")

In [ ]:
# Sanity check: verify targets behave hierarchically by recurrence category
# and that recovery_gap is 0 for pixels with no documented fire
meta_df_check = pd.DataFrame(meta)

print("Analytical target statistics by pixel type:")
print("=" * 80)
for ptype in config.PIXEL_TYPES:
    mask_pt = (meta_df_check["pixel_type"] == ptype).values
    n_pt = mask_pt.sum()
    if n_pt == 0:
        continue
    print(f"\n{ptype}  (n={n_pt})")
    for tname in ["recurrence", "severity", "persistence", "recovery_gap"]:
        arr = targets[tname][mask_pt]
        print(f"  {tname:14s}  mean={arr.mean():.3f}  "
              f"std={arr.std():.3f}  "
              f"median={np.median(arr):.3f}  "
              f"max={arr.max():.3f}")

# Sanity check: recovery_gap on no-fire pixels must be exactly 0
no_fire_mask = (meta_df_check["pixel_type"] == "no_fires").values
if no_fire_mask.any():
    rg_no_fires = targets["recovery_gap"][no_fire_mask]
    assert rg_no_fires.max() == 0.0, \
        f"BUG: recovery_gap should be 0 on no_fires, but max={rg_no_fires.max()}"
    print(f"\n✓ verified: recovery_gap==0 for all {no_fire_mask.sum()} pixels with no documented fire.")

## 5. Add dominant LC per pixel to the metadata

Downstream notebooks use this to stratify results by land-cover class.
The dominant LC is the mode of the current-year LC one-hot across the
pixel's entire time series, which handles the rare case where a
pixel changed class during the analysis period.

In [ ]:
lc_curr_cols = config.LC_FEATURES_CURR
lc_codes     = config.LC_CODES

# Aggregate all one-hot land-cover columns once. The previous groupby.apply
# invoked Python once per pixel and depended on a second full CSV copy.
lc_sums = (df.groupby("pixel_id", observed=True)[lc_curr_cols]
             .sum().reindex(pixel_ids))
dominant_lc = np.asarray(lc_codes, dtype=int)[
    lc_sums.to_numpy().argmax(axis=1)]

for i, m in enumerate(meta):
    m["lc_dominant"] = int(dominant_lc[i])

print("Pixel counts by dominant LC:")
uniq, cnt = np.unique(dominant_lc, return_counts=True)
for code, n in zip(uniq, cnt):
    print(f"  {code:3d}  {config.LC_CATEGORIES.get(int(code), 'unknown'):22s}  "
          f"{n:>6,}")

# No later cell needs the raw table or row mapping. Release them before the
# scaling step so they do not overlap with its temporary chunks.
del df, lc_sums, row_ok, i_ok, t_ok


## 6. Train / validation / test split

Random stratified split by pixel type. Stratification keeps each split
balanced across burn-recurrence categories. Non-random splits (spatial,
temporal, combined) are handled in Notebook 06.

In [ ]:
meta_df = pd.DataFrame(meta)
strat = meta_df["pixel_type"].apply(
    lambda t: "no_fires" if t == "no_fires" else "fires"
)

idx_all = np.arange(N)
idx_train, idx_temp = train_test_split(
    idx_all, test_size=1 - config.SPLITS["train_frac"],
    stratify=strat, random_state=config.SEED,
)
val_size = config.SPLITS["val_frac"] / (
    config.SPLITS["val_frac"] + config.SPLITS["test_frac"]
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=1 - val_size,
    stratify=strat.iloc[idx_temp], random_state=config.SEED,
)

print(f"Train: {len(idx_train)}   Val: {len(idx_val)}   Test: {len(idx_test)}")
for name, ix in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    print(f"  {name}: {meta_df.iloc[ix]['pixel_type'].value_counts().to_dict()}")

## 7. Scale spectral features only

The scaler is fit on training-data spectral features only. Land-cover
one-hot features stay at 0/1 so their semantic meaning is preserved.

In [ ]:
lc_feature_set = set(config.LC_FEATURES)
spectral_cols  = [i for i, name in enumerate(selected_features)
                  if name not in lc_feature_set]
lc_cols        = [i for i, name in enumerate(selected_features)
                  if name in lc_feature_set]

print(f"Spectral columns: {len(spectral_cols)}")
print(f"LC columns:       {len(lc_cols)}")

_spectral_idx = np.asarray(spectral_cols)
_lc_idx = np.asarray(lc_cols)
_time_idx = np.arange(T)
_n_spectral = len(spectral_cols)
_chunk_size = 25_000

# Fit and transform in bounded chunks. X itself becomes the scaled tensor;
# keeping a second 13+ GB X_scaled copy is unnecessary after targets are built.
if not globals().get("_X_SCALED_DONE", False):
    scaler = StandardScaler()
    for _start in range(0, len(idx_train), _chunk_size):
        _train_idx = idx_train[_start:_start + _chunk_size]
        _block = X[np.ix_(_train_idx, _time_idx, _spectral_idx)]
        scaler.partial_fit(_block.reshape(-1, _n_spectral))

    for _start in range(0, N, _chunk_size):
        _end = min(_start + _chunk_size, N)
        _block = X[_start:_end, :, _spectral_idx]
        _scaled = scaler.transform(
            _block.reshape(-1, _n_spectral)).astype(np.float32, copy=False)
        X[_start:_end, :, _spectral_idx] = _scaled.reshape(
            _end - _start, T, _n_spectral)

    del _train_idx, _block, _scaled
    _X_SCALED_DONE = True
else:
    print("X is already scaled in this kernel; keeping the existing scaler.")

X_scaled = X

# Compute diagnostics chunk-wise too; advanced indexing over all pixels would
# otherwise allocate another multi-gigabyte array just for these print lines.
_count = 0
_sum = 0.0
_sum_sq = 0.0
_lc_min = np.inf
_lc_max = -np.inf
for _start in range(0, N, _chunk_size):
    _end = min(_start + _chunk_size, N)
    _block = X_scaled[_start:_end, :, _spectral_idx]
    _sum += np.sum(_block, dtype=np.float64)
    _sum_sq += np.einsum("ijk,ijk->", _block, _block, dtype=np.float64)
    _count += _block.size
    if len(_lc_idx):
        _lc_block = X_scaled[_start:_end, :, _lc_idx]
        _lc_min = min(_lc_min, float(_lc_block.min()))
        _lc_max = max(_lc_max, float(_lc_block.max()))

_mean = _sum / _count
_std = np.sqrt(max(_sum_sq / _count - _mean ** 2, 0.0))
print(f"X_scaled shape: {X_scaled.shape}")
print(f"X_scaled spectral mean ~0: {_mean:.3f}, std ~1: {_std:.3f}")
if len(_lc_idx):
    print(f"X_scaled LC min: {_lc_min:.3f}, max: {_lc_max:.3f}")


## 8. Persist everything

In [ ]:
np.save(config.FILES["X_scaled"], X_scaled)
np.save(config.FILES["y_burned"], y)
np.savez(config.FILES["targets_aux"], **targets)
np.save(config.FILES["X_scaled"].replace(".npy", "_mask.npy"), mask)

with open(config.FILES["splits"], "w") as f:
    json.dump({
        "train": idx_train.tolist(),
        "val":   idx_val.tolist(),
        "test":  idx_test.tolist(),
    }, f)

with open(config.FILES["scaler"], "wb") as f:
    pickle.dump({
        "scaler":        scaler,
        "spectral_cols": spectral_cols,
        "lc_cols":       lc_cols,
    }, f)

with open(config.FILES["feature_names"], "w") as f:
    json.dump(selected_features, f)

meta_df.to_csv(config.FILES["X_scaled"].replace(".npy", "_meta.csv"),
               index=False)

print("All artefacts written to", config.PATHS["data_proc"])